In [2]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from llmsource import llm
from langgraph.checkpoint.sqlite import SqliteSaver

import sqlite3
model = "qwen3.5:9b"
 

# llm = ChatOllama(
#     model=model,
#     temperature=0
# )
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm.invoke(message)
    
    return{
        "messages":[response]
    }
    
connection = sqlite3.connect(database="chatbot.db",check_same_thread=False)    
checkpoint = SqliteSaver(connection)
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpoint)
threads = checkpoint.list(None)
print(threads)


<generator object SqliteSaver.list at 0x00000116F0033100>


In [20]:
def get_all_threads() -> list[str]:
    thread_ids = set()

    for checkpoint_tuple in checkpoint.list(None):
        thread_id = checkpoint_tuple.config["configurable"].get("thread_id")

        if thread_id:
            thread_ids.add(thread_id)

    return sorted(thread_ids)

In [21]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0,
    reasoning=False,
    num_ctx=4096,
    num_predict=512,
    keep_alive="30m"
)

In [ ]:

for chunk in llm.stream("what is 2368768 * 34363546"):
    print(chunk.content,end="",flush=True)

To find the product of $2,368,768$ and $34,363,546$, we can perform the multiplication:

$$2,368,768 \times 34,363,546 = 81,400,096,007,328$$

**Answer:**
**81,400,096,007,328**

In [5]:
for chunk in llm.stream("thank you"):
    print(chunk.content,end="",flush=True)

You're very welcome! 😊 Is there anything else I can help you with?

In [10]:
llm.invoke("thank you")

AIMessage(content="You're very welcome! 😊 Is there anything else I can help you with?", additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-08-26T11:40:51.7086926Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1501373200, 'load_duration': 1319000, 'prompt_eval_count': 14, 'prompt_eval_duration': 373658000, 'eval_count': 18, 'eval_duration': 1080644000, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--01a03ddf-769e-7283-b241-9e206e7e4e49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 18, 'total_tokens': 32})

In [30]:

init={"messages": [HumanMessage(content="what is 34 * 6587878 ")]}

response = llm.invoke(init)

# print(response['messages'][-1].content)      
for message in response["messages"]:
    print(type(message).__name__)
    print("content:", message.content)

    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)

    print("-" * 50)

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.

In [3]:
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, model='qwen3.5:9b', reasoning=False, num_ctx=4096, num_predict=512, temperature=0.0, keep_alive='30m')